# Case 2 · Is the sales bootcamp worth it? (Part 1)
**BUS-G 492 · Causal decisions with AI**

In [1]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case2_controls_training/data/"
for candidate in ["data/", "cases/case2_controls_training/data/", "../../../cases/case2_controls_training/data/", "../../../../cases/case2_controls_training/data/"]:
    if os.path.exists(candidate + "reps_2025.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

Reading data from: https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case2_controls_training/data/


### The data

`reps_2025.csv`: one row per sales rep, calendar year 2025

| Column | Meaning |
|---|---|
| `rep_id`, `region`, `segment` | rep and the customer segment they sell to |
| `tenure_years` | years at Northwind at the start of 2025 |
| `attended_bootcamp_2025` | 1 = attended the bootcamp in Feb–Mar 2025 |
| `client_meetings_2025` | client meetings logged in 2025 |
| `crm_logins_2025` | CRM logins in 2025 |
| `manager_rating_2025` | manager's year-end rating, 1–5 |
| `sales_2025` | bookings in 2025 (\$) |

In [2]:
import pandas as pd, numpy as np
df = pd.read_csv(DATA + "reps_2025.csv")
print(df.shape)
df.head()

(1200, 9)


,rep_id,region,segment,tenure_years,attended_bootcamp_2025,client_meetings_2025,crm_logins_2025,manager_rating_2025,sales_2025
0,R0001,Central,SMB,2.8,0,122,176,1.5,723200.0
1,R0002,Central,Mid-market,3.7,1,199,187,3.3,1253700.0
2,R0003,West,SMB,1.6,1,103,210,2.2,374600.0
3,R0004,West,Mid-market,4.3,0,138,175,1.5,813100.0
4,R0005,Central,Mid-market,1.8,0,173,163,3.0,937800.0


## Warm-up · Take a quick look at the data

Ask any AI assistant to write Python code that prints the answers to:

* How many reps attended the bootcamp, and what share of all reps is that?
* What were the average 2025 sales across all reps?

Copy the code into the cell below and run it. Got an error? Paste the error message back to the AI and try again.

**Prompt tip:** name the columns you need. For example: "I loaded a pandas DataFrame `df`. The column `attended_bootcamp_2025` is 1 if the rep attended and 0 if not, and `sales_2025` is the rep's 2025 sales. Write code that prints how many reps attended, what share that is, and the average of `sales_2025`."

In [3]:
# Your code here
# Reps who attended the bootcamp (1 = attended, 0 = did not)
n_attended = df["attended_bootcamp_2025"].sum()
n_total = len(df)
share = n_attended / n_total

print(f"Reps who attended the bootcamp: {n_attended} of {n_total}")
print(f"Share of all reps: {share:.1%}")

# Average 2025 sales across all reps
avg_sales = df["sales_2025"].mean()
print(f"Average 2025 sales: {avg_sales:,.2f}")

Reps who attended the bootcamp: 457 of 1200
Share of all reps: 38.1%
Average 2025 sales: 1,072,675.58


In [6]:
# Your code here
g = df.groupby("attended_bootcamp_2025")

table = pd.DataFrame({
    "N reps": g.size(),
    "% of reps": (g.size() / len(df) * 100).round(1),
    "Avg 2025 sales ($)": g["sales_2025"].mean().round(0),
    "Avg client meetings": g["client_meetings_2025"].mean().round(1),
    "Avg manager rating": g["manager_rating_2025"].mean().round(2),
}).T

table.columns = ["Did not attend", "Attended"]   # group 0 first, then group 1
table = table[["Attended", "Did not attend"]]    # match your table's order
print(table)

                       Attended  Did not attend
N reps                   457.00          743.00
% of reps                 38.10           61.90
Avg 2025 sales ($)   1383038.00       881780.00
Avg client meetings      195.70          156.10
Avg manager rating         2.99            1.61


## Stage 1 · The business problem

> **From:** Marcus Bell, Chief Revenue Officer, Northwind Software
> **Re:** Roll out the sales bootcamp?
>
> In early 2025 we offered a **two-week sales bootcamp**. It was voluntary; about 40% of our 1,200 reps went. It costs **\$15,000 per rep**, counting two weeks out of the field.
>
> Next year I can send the reps who haven't gone yet. I want a simple rule: **send every untrained rep with tenure ≤ T years** (tenure measured in January 2026, i.e. `tenure_years + 1`). T = 0 means nobody goes.
>
> Our gross margin on sales is **30%**. Tell me what T to use and what it will be worth.

## Stage 2 · Build the best model you can

Use any AI assistant you like. Start with this prompt, then keep going wherever it leads:

> *I have data on 1,200 B2B sales reps for 2025: region, segment, tenure, whether they attended a sales bootcamp, client meetings, CRM logins, manager rating, and sales. I need to determine the causal effect of the bootcamp on sales and decide which reps to send next year. Help me build the best model.*

- Try **more than one** approach, and compare them.
- If you build a predictive model, check it on data it hasn't seen.
- Keep a **prompt log** below: what you asked, and what you kept or rejected.

There is no single required method. Use your judgment.

### Prompt log

1.
2.
3.

In [7]:
# Your analysis. Add as many cells as you need.
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold

df = df.rename(columns={"attended_bootcamp_2025": "D", "sales_2025": "Y"})

# --- 0. Balance check ---
print(df.groupby("D")[["tenure_years"]].mean())
print(pd.crosstab(df["region"], df["D"], normalize="index"))
print(pd.crosstab(df["segment"], df["D"], normalize="index"))

# --- 1. Naive difference ---
print("Naive:", df.loc[df.D==1,"Y"].mean() - df.loc[df.D==0,"Y"].mean())

# --- 2. Regression adjustment (pre-treatment covariates only) ---
m2 = smf.ols("Y ~ D + tenure_years + C(region) + C(segment)", df).fit(cov_type="HC1")
print(m2.params["D"], m2.conf_int().loc["D"].values)

# --- 2b. Same, plus the possibly post-treatment variables (for comparison only) ---
m2b = smf.ols("Y ~ D + tenure_years + C(region) + C(segment) + client_meetings_2025"
              " + crm_logins_2025 + manager_rating_2025", df).fit(cov_type="HC1")
print(m2b.params["D"])

# --- 3. Does the effect vary with tenure? ---
m3 = smf.ols("Y ~ D*tenure_years + D*C(region) + D*C(segment) + tenure_years + C(region) + C(segment)",
             df).fit(cov_type="HC1")
print(m3.summary().tables[1])

# --- 4. Propensity score + overlap check ---
X = pd.get_dummies(df[["tenure_years","region","segment"]], drop_first=True).astype(float)
ps = LogisticRegression(max_iter=1000).fit(X, df.D).predict_proba(X)[:,1]
print(pd.Series(ps).groupby(df.D).describe())   # look for overlap

# --- 5. T-learner with out-of-sample (cross-fitted) predictions ---
tau = np.zeros(len(df))
for tr, te in KFold(5, shuffle=True, random_state=0).split(X):
    d, y = df.D.values, df.Y.values
    m1 = GradientBoostingRegressor(max_depth=2, n_estimators=150, learning_rate=0.05)
    m0 = GradientBoostingRegressor(max_depth=2, n_estimators=150, learning_rate=0.05)
    m1.fit(X.iloc[tr][d[tr]==1], y[tr][d[tr]==1])
    m0.fit(X.iloc[tr][d[tr]==0], y[tr][d[tr]==0])
    tau[te] = m1.predict(X.iloc[te]) - m0.predict(X.iloc[te])
df["tau_hat"] = tau

# --- 6. Value of each rule T (untrained reps only) ---
def value(T, tau_col="tau_hat"):
    sel = (df.D == 0) & (df.tenure_years + 1 <= T)
    return ((0.30 * df.loc[sel, tau_col] - 15000).sum(), sel.sum())

for T in range(0, int(df.tenure_years.max()) + 3):
    v, n = value(T)
    print(f"T={T}: send {n} reps, net value ${v:,.0f}")

   tenure_years
D              
0      3.725303
1      4.108315
D                     0         1
region                           
Central        0.620130  0.379870
East           0.591036  0.408964
International  0.625000  0.375000
West           0.642667  0.357333
D                  0         1
segment                       
Enterprise  0.447471  0.552529
Mid-market  0.617577  0.382423
SMB         0.704981  0.295019
Naive: 501258.0440051715
281726.7672777971 [227865.98622772 335587.54832787]
-212187.40270339788
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                     1.715e+06   6.56e+04     26.151      0.000    1.59e+06    1.84e+06
C(region)[T.East]             1.672e+04   3.27e+04      0.512      0.609   -4.73e+04    8.08e+04
C(region)[T.International]    1.922e+04   4.49e+04      0.428      0.668   -6.87e+04    1.07

In [8]:
# Effect by segment with confidence intervals
for seg in ["Enterprise", "Mid-market", "SMB"]:
    test = f"D + D:C(segment)[T.{seg}]" if seg != "Enterprise" else "D"
    r = m3.t_test(test)
    print(seg, r.effect, r.conf_int())

# Break-even check per segment: need effect > 50,000
print(df[df.D==0].groupby("segment").size())   # untrained reps by segment


Enterprise [493514.44767306] [[276372.13563333 710656.75971279]]
Mid-market [245816.3911895] [[ 77432.49132469 414200.29105431]]
SMB [208300.96958007] [[ 54642.45240644 361959.48675369]]
segment
Enterprise    115
Mid-market    260
SMB           368
dtype: int64


In [9]:
rng = np.random.default_rng(0)
vals = []
for b in range(200):
    s = df.sample(len(df), replace=True, random_state=int(rng.integers(1e9))).reset_index(drop=True)
    m = smf.ols("Y ~ D*C(segment) + tenure_years + C(region)", s).fit()
    untr = s[s.D == 0]
    eff = m.predict(untr.assign(D=1)) - m.predict(untr.assign(D=0))
    vals.append((0.30 * eff - 15000).sum())
print(np.percentile(vals, [2.5, 50, 97.5]))

[36481610.22607986 46004816.00491011 57687944.39210219]


## Stage 3 · Commit to a decision

**Predicted profit** = 30% × (extra 2026 sales caused by the bootcamp, for the reps you send) − \$15,000 × (number of reps you send). Only reps who did *not* attend in 2025 are eligible.

**Every number must come from your analysis above.** Your recommendation will be deployed, and you'll see what actually happens.

In [ ]:
TEAM = ""                  # e.g. "Team 3: Ana & Ben"  (use the same name every time)
MAX_TENURE = None          # send untrained reps with January-2026 tenure <= this; 0 = nobody  (e.g. 3)
PREDICTED_PROFIT = None    # in dollars, as a number, e.g. 150000
JUSTIFICATION = """
Three sentences for the decision maker: what you found, what you recommend, and why.
"""
CONFIDENCE = """
What is the single most likely reason your recommendation could be wrong?
"""

In [ ]:
# Check your submission, then enter what it prints in the submission form.
# Stops with a message if anything is still blank, so you don't submit an empty field.
blank = [name for name, value in [("TEAM", TEAM), ("MAX_TENURE", MAX_TENURE), ("PREDICTED_PROFIT", PREDICTED_PROFIT)] if value in (None, "")]
if blank:
    raise ValueError("Fill these in before submitting: " + ", ".join(blank))
PREDICTED_PROFIT = float(PREDICTED_PROFIT)   # must be a number
print("Enter exactly these values in the submission form:")
print()
print(f"team:              {TEAM}")
print("case:              2")
print("round:             1  (Part I)")
print(f"decision:          {MAX_TENURE}")
print(f"predicted_profit:  {PREDICTED_PROFIT:.0f}")

## Submit (round 1)

**[Open the submission form](https://docs.google.com/forms/d/e/1FAIpQLSf-87eDsEHRVat_a4le91sC7bfZFel9fAHNmMtHVuGJzqaHOQ/viewform)** and enter the values printed above, exactly as shown.

- **Predicted profit:** a number such as `150000` (`$150,000` and `150K` also work, but not words).
- **Case:** `2`.  **Round:** `1` (Part I).
- Check the printed values before you submit. If your team submits more than once for the same round, only the **last** submission counts.
- Use the **same team name** in every round.